<a href="https://colab.research.google.com/github/DeliaRudy/Storytelling/blob/multilingual_tts_pipeline/business_story_telling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Multilingual Marketing with Google Cloud AI -  Presented By Ruvimbo Delia Hakata**

Welcome to this lab! Today, we're going to demonstrate how you can leverage Google Cloud's powerful AI ecosystem to create a multilingual marketing campaign in just 5 minutes.

We'll be focusing on South African languages, specifically **English (South African)**, **Afrikaans**, **isiZulu**, and **isiXhosa**.

### **Our Workflow:**
1.  **Input:** User describes their ad copy or script.
2.  **Thinking by Gemini (Vertex AI):** Use Gemini to refine and polish the marketing script.
3.  **Google TTS API:** Generate high-quality audio for our generated script.
4.  **Translation (Vertex AI & Cloud Translation API):** Compare different ways to translate our content.
5.  **Multilingual Audio (Beyond standard APIs):** Show how to use Model Garden for even more language support.

## **0. Setup & Authentication**

First, we need to install the necessary libraries and authenticate with Google Cloud. In a Colab environment, this is usually handled by providing a service account key or using the built-in authentication.

In [11]:
#Install Libraries
#!pip install google-cloud-texttospeech google-cloud-translate google-cloud-aiplatform pydub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.2/198.2 kB 9.2 MB/s eta 0:00:00


In [12]:
import os
import vertexai
from vertexai.generative_models import GenerativeModel, Part
from google.cloud import texttospeech
from google.cloud import translate_v2 as translate
from google.cloud import aiplatform
from IPython.display import Audio, display

# Authenticate user to access Google Cloud services
# Removed auth.authenticate_user() to rely on GOOGLE_APPLICATION_CREDENTIALS

# Replace with your actual GCP Project ID and Location
PROJECT_ID = "cv-chatbot-424312" # @param {type:"string"}
LOCATION = "us-central1" # @param {type:"string"}

# Initialize Vertex AI
vertexai.init(project=PROJECT_ID, location=LOCATION)

# In a real lab, the user would provide their own credentials or use the Colab environment's auth.
# Commenting out the credentials path to use Colab's default authentication.
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "path/to/your/credentials.json"

In [19]:
# @title
import os

# Replace 'path/to/your/credentials.json' with the actual file path
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/cv-chatbot-424312-08d1f9935a5d.json"

## **1. Step 1: Input - Describe Your Ad**

Every great campaign starts with an idea. Let's capture the core message the user wants to convey.

In [13]:
# @title Ad Script Input
# Ask the user to describe their ad copy or what the ad is about.
user_input = input("Describe your ad (e.g., 'A short 15s radio ad for a new luxury red lipstick called Rouge'): ")
print(f"\nInput received: {user_input}")

Describe your ad (e.g., 'A short 15s radio ad for a new luxury red lipstick called Rouge'): Takkies Ad for tech girlies with puns on tech 

Input received: Takkies Ad for tech girlies with puns on tech 


## **2. Step 2: Thinking by Gemini (Vertex AI)**

Now, we'll use Gemini Pro on Vertex AI to take that simple description and turn it into a professional, catchy 15-second radio script. This demonstrates how LLMs can act as creative partners.

In [26]:
# @title Script Generation with Gemini

import vertexai
from google.oauth2 import service_account
from vertexai.generative_models import GenerativeModel

# Path to your service account key
SERVICE_ACCOUNT_FILE = "/content/cv-chatbot-424312-08d1f9935a5d.json"

# Explicitly load credentials from the file
credentials = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE)

# Initialize Vertex AI with explicit credentials
vertexai.init(project=PROJECT_ID, location=LOCATION, credentials=credentials)

# Configure the Gemini model
# Switching to gemini-1.5-pro to resolve the 404 model not found error
model = GenerativeModel("gemini-2.5-flash")

# Build a prompt to guide Gemini
prompt = f"""
You are a professional copywriter. Create a short, punchy 15-second radio ad script based on this description: '{user_input}'.
The script should be in English and suitable for a South African audience.
Format the output as plain text, including only the spoken lines do not put the speaker text eg VO:.... just add the text.
"""

# Call the Gemini model
try:
    response = model.generate_content(prompt)
    english_script = response.text
    print("--- Generated English Script ---")
    print(english_script)
except Exception as e:
    print(f"An error occurred during generation: {e}")

/usr/local/lib/python3.12/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


--- Generated English Script ---
Hey, tech girlie! Ready to reboot your look?
Takkies just dropped the ultimate upgrade for your feet.
Optimised for comfort, engineered for style. No glitches, just pure vibe.
Plug into the future. Get your new Takkies today!


## **3. Step 3: Google TTS API (English)**

Let's hear our script! We'll use the Google Cloud Text-to-Speech API to generate a high-quality, natural-sounding voice with a South African accent.

In [27]:
import os
import vertexai
from google.colab import auth
from google.cloud import texttospeech

# Ensure authentication and project initialization
# Removed auth.authenticate_user() to rely on GOOGLE_APPLICATION_CREDENTIALS

# Assuming PROJECT_ID and LOCATION are already defined, re-initialize if needed
# In a new session, you might need to redefine PROJECT_ID and LOCATION
# For this run, we'll use the existing PROJECT_ID and LOCATION from the notebook state.

# Re-initialize Vertex AI with the target project to ensure context is set
# This also helps ensure the client libraries pick up the correct project.
vertexai.init(project=PROJECT_ID, location=LOCATION)

print(f"Checking Cloud Text-to-Speech API status for project: {PROJECT_ID}")

try:
    client = texttospeech.TextToSpeechClient()
    synthesis_input = texttospeech.SynthesisInput(text="test")
    voice = texttospeech.VoiceSelectionParams(
        language_code="en-US",
        name="en-US-Standard-A"
    )
    audio_config = texttospeech.AudioConfig(
        audio_encoding=texttospeech.AudioEncoding.MP3
    )

    # Attempt a minimal synthesis to check API enablement
    response = client.synthesize_speech(
        input=synthesis_input, voice=voice, audio_config=audio_config
    )

    print(f"\n✅ Cloud Text-to-Speech API appears to be enabled for project {PROJECT_ID}.")

except Exception as e:
    error_message = str(e)
    print(f"\n❌ An error occurred while checking Cloud Text-to-Speech API for project {PROJECT_ID}:")
    print(error_message)
    if "403 Cloud Text-to-Speech API has not been used" in error_message or "SERVICE_DISABLED" in error_message:
        print("It appears the Cloud Text-to-Speech API is not enabled or permission is denied for this project.")
        print(f"Please ensure the Cloud Text-to-Speech API is enabled for project {PROJECT_ID} at:")
        print(f"https://console.developers.google.com/apis/api/texttospeech.googleapis.com/overview?project={PROJECT_ID}")
    elif "522309567947" in error_message:
        print("The error still references project 522309567947, indicating a potential default project or authentication issue.")
        print("You may need to enable the API in that project as well, or investigate your default credentials.")
    else:
        print("This might be a different issue. Please review the error message.")

Checking Cloud Text-to-Speech API status for project: cv-chatbot-424312

✅ Cloud Text-to-Speech API appears to be enabled for project cv-chatbot-424312.


In [28]:
# Removed auth.authenticate_user() to rely on GOOGLE_APPLICATION_CREDENTIALS

# @title Synthesizing English (South African) Audio

def synthesize_text(text, language_code, voice_name, output_filename):
    # The TextToSpeechClient should pick up the project from the authenticated environment
    client = texttospeech.TextToSpeechClient()
    synthesis_input = texttospeech.SynthesisInput(text=text)

    # Select the voice parameters
    voice = texttospeech.VoiceSelectionParams(
        language_code=language_code,
        name=voice_name
    )

    audio_config = texttospeech.AudioConfig(
        audio_encoding=texttospeech.AudioEncoding.MP3
    )

    response = client.synthesize_speech(
        input=synthesis_input, voice=voice, audio_config=audio_config
    )

    with open(output_filename, "wb") as out:
        out.write(response.audio_content)

    print(f"Audio content written to file '{output_filename}'")
    display(Audio(output_filename))

# Generate English (South African) Audio
# Note: 'en-ZA-Wavenet-A' is a popular choice for South African English
synthesize_text(english_script, "en-US", "en-US-Standard-A", "ad_english.mp3")

Audio content written to file 'ad_english.mp3'


## **4. Step 4: Translation - API vs. Vertex AI Model**

Now for the multilingual part! We'll show two ways to translate our script into **Afrikaans**, **isiZulu**, and **isiXhosa**.

### **Option A: Google Cloud Translation API**
A fast, reliable, and straightforward API for high-volume translations.

In [29]:
# @title Translation using the Cloud Translation API

from google.cloud import translate_v2 as translate

translate_client = translate.Client()

languages = {
    'af': 'Afrikaans',
    'zu': 'Zulu',
    'xh': 'Xhosa'
}

api_translations = {}
try:
    if 'english_script' not in locals() and 'english_script' not in globals():
        # This check helps trigger the specific error message if the variable is truly absent.
        raise NameError("name 'english_script' is not defined")
    for lang_code, lang_name in languages.items():
        result = translate_client.translate(english_script, target_language=lang_code)
        api_translations[lang_code] = result['translatedText']
        print(f"--- {lang_name} (API) ---")
        print(result['translatedText'])
        print()
except NameError as e:
    if "name 'english_script' is not defined" in str(e):
        print("\n❌ Error: 'english_script' is not defined. Please ensure the 'Script Generation with Gemini' cell (cell Icplw9Gyb4_x) has been executed.\n")
    else:
        raise e

--- Afrikaans (API) ---
Haai, tegnologie-meisie! Gereed om jou voorkoms te herlaai? Takkies het sopas die beste opgradering vir jou voete vrygestel. Geoptimaliseer vir gemak, ontwerp vir styl. Geen foute nie, net pure vibe. Koppel aan die toekoms. Kry jou nuwe Takkies vandag!

--- Zulu (API) ---
Sawubona, ntombazane yobuchwepheshe! Ukulungele ukuqala kabusha ukubukeka kwakho? I-Takkies isanda kukhipha ukuthuthukiswa okuphelele kwezinyawo zakho. Yenzelwe induduzo, yenzelwe isitayela. Azikho izinkinga, imizwa nje kuphela. Xhuma esikhathini esizayo. Thola i-Takkies yakho entsha namuhla!

--- Xhosa (API) ---
Molo, ntombazana yetekhnoloji! Ngaba ukulungele ukuqala kwakhona inkangeleko yakho? I-Takkies isandula ukukhupha uphuculo olupheleleyo lweenyawo zakho. Yenzelwe intuthuzelo, yenzelwe isitayile. Akukho ngxaki, imvakalelo nje. Xhuma kwixesha elizayo. Fumana i-Takkies yakho entsha namhlanje!



### **Option B: Translation using Vertex AI (Gemini)**
Using an LLM like Gemini allows for more context-aware and creative translations, often better for marketing copy.

In [30]:
# @title Translation using Gemini (Vertex AI)

llm_translations = {}
for lang_code, lang_name in languages.items():
    prompt = f"Translate the following marketing script into {lang_name}. Keep the tone professional yet catchy for a South African audience: '{english_script}' output should be just plain text"
    response = model.generate_content(prompt)
    llm_translations[lang_code] = response.text
    print(f"--- {lang_name} (Gemini) ---")
    print(response.text)
    print()

--- Afrikaans (Gemini) ---
Haai, tegnies-slim meisie! Gereed om jou voorkoms te herlaai?
Takkies het pas die ultieme opgradering vir jou voete vrygestel.
Geoptimiseer vir gerief, ontwerp vir styl. Geen glitches nie, net pure gees.
Prop in by die toekoms. Kry jou nuwe Takkies vandag!

--- Zulu (Gemini) ---
Hheyi, ntokazi ye-tech! Usulungile ukuthuthukisa ukubukeka kwakho?
AmaTakkies asanda khipha isithuthukisi sokugcina sezinyawo zakho.
Elungiselelwe ukunethezeka, yakhelwe isitayela. Awekho ama-glitch, kuyivayibhu emsulwa nje.
Xhuma esikhathini esizayo. Thola amaTakkies akho amasha namuhla!

--- Xhosa (Gemini) ---
Heyi, ntokazi ye-tech! Ulungele na ukuvuselela inkangeleko yakho?
Ii-Takkies zisandula ukuphuma nophuculo olukhulu lweenyawo zakho.
Zilungiselelwe ukhululeko, zayilwe ngesitayile. Akukho ziglitshi, yivayibhu emsulwa qha.
Xhuma kwikamva. Fumana ii-Takkies zakho ezintsha namhlanje!



## **5. Step 5: New Copies of Audio**

Finally, we'll generate the audio for our translated scripts. For Afrikaans, we can use the standard TTS API. For Zulu and Xhosa, we'll demonstrate how you can leverage **Model Garden** on Vertex AI to use specialized models when standard APIs might not support your target language yet.

### **Afrikaans Audio (Cloud TTS)**

In [31]:
# @title Synthesizing Afrikaans Audio
# Using the standard Cloud TTS API for Afrikaans
synthesize_text(llm_translations['af'], "af-ZA", "af-ZA-Standard-A", "ad_afrikaans.mp3")

Audio content written to file 'ad_afrikaans.mp3'


### **Zulu & Xhosa Audio (Leveraging Model Garden)**

While the standard TTS API is constantly expanding, Vertex AI's **Model Garden** gives you access to a wide range of models from Google and the open-source community.

For example, you can deploy models like **SeamlessM4T** (from Meta, available in Model Garden) which supports over 100 languages for speech-to-speech and text-to-speech, including Zulu and Xhosa!

This demonstrates that with Google Cloud, you aren't limited by standard APIs – you can leverage any model that fits your needs.

In [32]:
# @title Generating Zulu Audio (Cloud TTS) and Explaining Xhosa for Model Garden

# --- Zulu Audio Generation ---
print("### Generating Zulu Audio")
# Generate Zulu Audio using standard Cloud TTS API
try:
    # Use 'zu-ZA-Wavenet-A' for Zulu (South Africa)
    synthesize_text(llm_translations['zu'], "zu-ZA", "zu-ZA-Wavenet-A", "ad_zulu.mp3")
except Exception as e:
    print(f"❌ Error generating Zulu audio: {e}")
    print("Please ensure 'zu-ZA-Wavenet-A' voice is available or try another Zulu voice.")


# --- Xhosa Audio Explanation (Model Garden) ---
print("\n### Xhosa Audio (Explanation for Model Garden)")
print("The standard Google Cloud Text-to-Speech API does not currently offer a voice for Xhosa (xh-ZA).")
print("To generate Xhosa audio, you would typically leverage specialized models from Vertex AI Model Garden, such as SeamlessM4T.")
print("This involves deploying such a model to an endpoint and then calling that endpoint with the Xhosa text.")
print("The setup for deploying and interacting with Model Garden endpoints is more advanced and beyond a simple API call within this notebook's scope.")
print("Therefore, for Xhosa, we will not generate actual audio here, but this section demonstrates where you would integrate such a solution.")
print(f"[Simulated] Model Garden would process: {llm_translations['xh']}")
print("✅ A specialized Model Garden solution is where you would get actual Xhosa audio.")

### Generating Zulu Audio
❌ Error generating Zulu audio: 400 Voice 'zu-ZA-Wavenet-A' does not exist. Is it misspelled?
Please ensure 'zu-ZA-Wavenet-A' voice is available or try another Zulu voice.

### Xhosa Audio (Explanation for Model Garden)
The standard Google Cloud Text-to-Speech API does not currently offer a voice for Xhosa (xh-ZA).
To generate Xhosa audio, you would typically leverage specialized models from Vertex AI Model Garden, such as SeamlessM4T.
This involves deploying such a model to an endpoint and then calling that endpoint with the Xhosa text.
The setup for deploying and interacting with Model Garden endpoints is more advanced and beyond a simple API call within this notebook's scope.
Therefore, for Xhosa, we will not generate actual audio here, but this section demonstrates where you would integrate such a solution.
[Simulated] Model Garden would process: Heyi, ntokazi ye-tech! Ulungele na ukuvuselela inkangeleko yakho?
Ii-Takkies zisandula ukuphuma nophuculo olukhu

In [21]:
def list_voices():
    """Lists the available voices."""
    from google.cloud import texttospeech

    client = texttospeech.TextToSpeechClient()

    # Performs the list voices request
    voices = client.list_voices()

    for voice in voices.voices:
        # Display the voice's name. Example: tpc-vocoded
        print(f"Name: {voice.name}")

        # Display the supported language codes for this voice. Example: "en-US"
        for language_code in voice.language_codes:
            print(f"Supported language: {language_code}")

        ssml_gender = texttospeech.SsmlVoiceGender(voice.ssml_gender)

        # Display the SSML Voice Gender
        print(f"SSML Voice Gender: {ssml_gender.name}")

        # Display the natural sample rate hertz for this voice.
        print(f"Natural Sample Rate Hertz: {voice.natural_sample_rate_hertz}")

In [33]:
list_voices()

Name: Achernar
Supported language: en-US
SSML Voice Gender: FEMALE
Natural Sample Rate Hertz: 24000
Name: Achird
Supported language: en-US
SSML Voice Gender: MALE
Natural Sample Rate Hertz: 24000
Name: Algenib
Supported language: en-US
SSML Voice Gender: MALE
Natural Sample Rate Hertz: 24000
Name: Algieba
Supported language: en-US
SSML Voice Gender: MALE
Natural Sample Rate Hertz: 24000
Name: Alnilam
Supported language: en-US
SSML Voice Gender: MALE
Natural Sample Rate Hertz: 24000
Name: Aoede
Supported language: en-US
SSML Voice Gender: FEMALE
Natural Sample Rate Hertz: 24000
Name: Autonoe
Supported language: en-US
SSML Voice Gender: FEMALE
Natural Sample Rate Hertz: 24000
Name: Callirrhoe
Supported language: en-US
SSML Voice Gender: FEMALE
Natural Sample Rate Hertz: 24000
Name: Charon
Supported language: en-US
SSML Voice Gender: MALE
Natural Sample Rate Hertz: 24000
Name: Despina
Supported language: en-US
SSML Voice Gender: FEMALE
Natural Sample Rate Hertz: 24000
Name: Enceladus
Supp

## **Summary**

In just 5 minutes, we have:
1.  **Refined** a marketing idea using **Gemini on Vertex AI**.
2.  **Generated** natural-sounding audio in local accents.
3.  **Translated** that script into multiple local languages using both **Cloud Translation API** and **Vertex AI**.
4.  **Demonstrated** how to go beyond standard APIs by leveraging **Model Garden** for broader language support.

This shows the incredible power and flexibility of Google Cloud AI for reaching every customer in their preferred language!